# 05 — Model Evaluation and Interpretation
**Research question:** Did the regularization study improve held-out validation performance, and how reliably does the frozen selected configuration generalize to unseen speakers?

This notebook loads the chosen train-fitted pipeline and the original-best comparison from 04. It reports regularization diagnostics, learning curves, class-wise ROC/PR, permutation importance, and a **single descriptive test evaluation** after refit on train+validation. Test actors have been inspected historically; do not tune further using their outcomes.

## 1. Load the frozen model and experiment

In [ ]:
from pathlib import Path
import hashlib, json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap
from IPython.display import display
from google.colab import drive
from sklearn.base import clone
from sklearn.model_selection import GroupKFold, learning_curve
from sklearn.inspection import permutation_importance
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             confusion_matrix, roc_curve, auc,
                             precision_recall_curve, average_precision_score,
                             classification_report)
from sklearn.preprocessing import label_binarize

if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive')
SEED = 42
PROJECT_DIR = Path('/content/drive/MyDrive/speech-emotion-recognition-project-d')
FEATURES_PATH = PROJECT_DIR/'data/processed/ravdess_features.csv'
BASE = PROJECT_DIR/'results/04_ml'
CONFIG_PATH = BASE/'metrics/frozen_config.json'
MODEL_PATH = BASE/'validation_selected_pipeline.joblib'
BASELINE_MODEL_PATH = BASE/'original_baseline_pipeline.joblib'
COMPARISON_PATH = BASE/'metrics/all_model_comparison.csv'
RESULTS_DIR = PROJECT_DIR/'results/05_evaluation'
FIGURES_DIR = RESULTS_DIR/'figures'
for path in (FEATURES_PATH, CONFIG_PATH, MODEL_PATH, BASELINE_MODEL_PATH, COMPARISON_PATH):
    if not path.is_file(): raise FileNotFoundError(f'Missing output of notebook 03/04: {path}')
for path in (RESULTS_DIR,FIGURES_DIR): path.mkdir(parents=True,exist_ok=True)
with CONFIG_PATH.open() as file: config=json.load(file)
FEATURE_COLS=config['feature_names']
EMOTIONS=['neutral','calm','happy','sad','angry','fearful','disgust','surprised']
model_train=joblib.load(MODEL_PATH)  # Only load your own trusted joblib file.
original_model=joblib.load(BASELINE_MODEL_PATH)
model_comparison=pd.read_csv(COMPARISON_PATH)
if config.get('fast_mode'):
    raise RuntimeError('Notebook 04 ran in FAST_MODE; rerun 04 with FAST_MODE = False before evaluation.')
df=pd.read_csv(FEATURES_PATH).sort_values('relative_path').reset_index(drop=True)
assert hashlib.sha256(FEATURES_PATH.read_bytes()).hexdigest()==config['data_sha256']
assert len(df)==1440 and df.relative_path.is_unique and len(FEATURE_COLS)==39
actors=df.actor_id.astype(int)
train_mask=actors.isin(config['train_actors'])
val_mask=actors.isin(config['validation_actors'])
test_mask=actors.isin(config['test_actors'])
assert (sum(train_mask),sum(val_mask),sum(test_mask))==(720,360,360)
X_train=df.loc[train_mask,FEATURE_COLS].reset_index(drop=True)
y_train=df.loc[train_mask,'emotion'].reset_index(drop=True)
g_train=actors.loc[train_mask].reset_index(drop=True)
X_val=df.loc[val_mask,FEATURE_COLS].reset_index(drop=True)
y_val=df.loc[val_mask,'emotion'].reset_index(drop=True)
assert set(model_train.classes_) == set(EMOTIONS)
print('Frozen selection:',config['selected_model'])
print('Data checksum verified; held-out actors are not used for model tuning.')

In [ ]:
PURPLE, BLUE, DARK_PURPLE = '#B57EDC','#2A52BE','#9461B5'
PALE='#FAF9FC'
SEQ = LinearSegmentedColormap.from_list('lavender',['#FFFFFF','#E4CBF2',PURPLE,DARK_PURPLE])
DIVERGING=LinearSegmentedColormap.from_list('blue_lavender',[BLUE,PALE,PURPLE])
sns.set_theme(style='white',font_scale=.94)
def style_ax(ax,title,grid='y'):
    ax.set_title(title,loc='left',fontweight='bold',pad=13)
    ax.grid(axis=grid,ls='--',alpha=.18)
    ax.set_axisbelow(True)
    ax.tick_params(axis='both',length=0,labelrotation=0)
    sns.despine(ax=ax)
def save_plot(fig,name):
    fig.savefig(FIGURES_DIR/name,dpi=170,bbox_inches='tight')
    plt.show()

## 2. Regularization: has the chosen model improved?

The **fixed validation** comparison is the model-selection evidence. A gap between training and held-out performance is diagnostic, not an optimization objective by itself. If the original baseline wins, it remains the selected model; the regularized configurations are reported as informative negative results.

In [ ]:
# Do not refit or select a model here; both pipelines are already fitted on training actors in 04.
base_pred = original_model.predict(X_val)
selected_pred = model_train.predict(X_val)
validation_check = pd.DataFrame([
    {'Configuration':'Original best',
     'Model':config['original_baseline_model'],
     'Validation Macro F1':f1_score(y_val, base_pred, labels=EMOTIONS,
                                   average='macro', zero_division=0)},
    {'Configuration':'Selected after regularization study',
     'Model':config['selected_model'],
     'Validation Macro F1':f1_score(y_val, selected_pred, labels=EMOTIONS,
                                   average='macro', zero_division=0)}
])
assert abs(validation_check.iloc[0]['Validation Macro F1'] -
           config['original_baseline_validation_f1']) < 1e-8
assert abs(validation_check.iloc[1]['Validation Macro F1'] -
           config['selected_validation_f1']) < 1e-8

display(validation_check.round(3))
comparison_cols=['model','train_f1','cv_f1','cv_sd','validation_f1','gap_train_cv']
display(model_comparison[comparison_cols].round(3))
print('Selected − original validation Macro F1:',
      f"{validation_check.iloc[1]['Validation Macro F1'] - validation_check.iloc[0]['Validation Macro F1']:+.3f}")

## 3. Learning curve (train-only grouped CV)
A diagnostic for potential overfitting/data limitation, not a retuning criterion. Prefix-based training subsets can introduce speaker-composition variation.

In [ ]:
# Speaker-grouped learning curve; train-only (actors 01–12), fixed model parameters.
from sklearn.base import clone
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score

speaker_counts = [2, 4, 6, 8]
train_scores = np.zeros((len(speaker_counts), 3))
cv_scores = np.zeros_like(train_scores)

cv = GroupKFold(n_splits=3)

for fold, (train_idx, val_idx) in enumerate(
    cv.split(X_train, y_train, groups=g_train)
):
    speakers = np.random.default_rng(SEED + fold).permutation(
        np.unique(g_train.iloc[train_idx])
    )

    for i, n in enumerate(speaker_counts):
        selected = train_idx[
            g_train.iloc[train_idx].isin(speakers[:n]).to_numpy()
        ]

        model = clone(model_train)
        model.fit(X_train.iloc[selected], y_train.iloc[selected])

        train_scores[i, fold] = f1_score(
            y_train.iloc[selected],
            model.predict(X_train.iloc[selected]),
            average="macro"
        )

        cv_scores[i, fold] = f1_score(
            y_train.iloc[val_idx],
            model.predict(X_train.iloc[val_idx]),
            average="macro"
        )

train_mean = train_scores.mean(axis=1)
cv_mean = cv_scores.mean(axis=1)
cv_sd = cv_scores.std(axis=1)

fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(
    speaker_counts, train_mean,
    "o--", color="#2A52BE", linewidth=1.8,
    label="Training"
)

ax.plot(
    speaker_counts, cv_mean,
    "o-", color="#B57EDC", linewidth=2.5,
    label="Grouped CV"
)

ax.fill_between(
    speaker_counts,
    np.maximum(cv_mean - cv_sd, 0),
    np.minimum(cv_mean + cv_sd, 1),
    color="#B57EDC", alpha=0.15,
    label="CV ± 1 SD"
)

ax.set(
    xlabel="Number of training speakers",
    ylabel="Macro F1",
    xticks=speaker_counts,
    ylim=(0, 1.05)
)

ax.set_title(
    "Learning curves across training speakers",
    fontweight="bold", loc="left", pad=15
)

ax.grid(axis="y", linestyle="--", alpha=0.2)
ax.set_axisbelow(True)
ax.legend(frameon=False, loc="center right")
sns.despine()
plt.tight_layout()

fig.savefig(
    FIGURES_DIR / "learning_curve.png",
    dpi=170, bbox_inches="tight"
)
plt.show()

learning_df = pd.DataFrame({
    "Training speakers": speaker_counts,
    "Train Macro F1": train_mean,
    "CV Macro F1": cv_mean,
    "CV SD": cv_sd,
    "Generalization gap": train_mean - cv_mean
})

display(learning_df.round(3))
learning_df.to_csv(RESULTS_DIR / "learning_curve.csv", index=False)

## 4. ROC-AUC and precision–recall on fixed validation
One-vs-rest ROC-AUC and Average Precision summarize ranking performance for each emotion. The table is the primary comparison; an interactive diagnostic below lets the reader inspect one class and curve at a time. The dashed PR baseline equals that emotion's prevalence. These are descriptive results, not a basis for changing the frozen model or optimizing thresholds.

In [ ]:
# Scores are required for ROC/PR; predicted class labels are insufficient.
from ipywidgets import Dropdown, HBox, Output

def score_matrix(estimator, X, labels):
    if hasattr(estimator, 'predict_proba'):
        scores = estimator.predict_proba(X)
    elif hasattr(estimator, 'decision_function'):
        scores = estimator.decision_function(X)
    else:
        raise TypeError('Selected model exposes no probability or decision scores.')
    classes = list(estimator.classes_)
    return np.asarray(scores)[:, [classes.index(label) for label in labels]]


def roc_pr_summary(y_true, scores, prefix):
    binary = label_binarize(y_true, classes=EMOTIONS)
    rows = []
    for i, label in enumerate(EMOTIONS):
        fpr, tpr, _ = roc_curve(binary[:, i], scores[:, i])
        rows.append({
            'emotion': label,
            'ROC_AUC_OVR': auc(fpr, tpr),
            'average_precision': average_precision_score(binary[:, i], scores[:, i]),
            'prevalence': binary[:, i].mean()
        })
    result = pd.DataFrame(rows)
    result.to_csv(RESULTS_DIR / f'{prefix.lower()}_roc_pr.csv', index=False)
    display(result.round(3).set_index('emotion'))
    print(f'{prefix} macro ROC-AUC: {result.ROC_AUC_OVR.mean():.3f}; macro AP: {result.average_precision.mean():.3f}')
    return result


def explore_roc_pr(y_true, scores, prefix):
    binary = label_binarize(y_true, classes=EMOTIONS)
    emotion_dropdown = Dropdown(options=EMOTIONS, value='angry', description='Emotion:')
    curve_dropdown = Dropdown(options=['ROC', 'Precision–Recall'], value='Precision–Recall', description='Curve:')
    chart = Output()

    def update(_=None):
        idx = EMOTIONS.index(emotion_dropdown.value)
        label = emotion_dropdown.value
        truth, score = binary[:, idx], scores[:, idx]
        fig, ax = plt.subplots(figsize=(8.8, 5))
        if curve_dropdown.value == 'ROC':
            x, y, _ = roc_curve(truth, score)
            metric = auc(x, y)
            ax.plot(x, y, color=BLUE, lw=2.4, label=f'ROC-AUC = {metric:.3f}')
            ax.plot([0, 1], [0, 1], ls='--', lw=1.2, color='#A6BCCB', label='Random ranking')
            ax.set(xlabel='False positive rate', ylabel='True positive rate')
        else:
            precision, recall, _ = precision_recall_curve(truth, score)
            metric = average_precision_score(truth, score)
            ax.step(recall, precision, where='post', color=PURPLE, lw=2.4,
                    label=f'Average precision = {metric:.3f}')
            prevalence = truth.mean()
            ax.axhline(prevalence, color=BLUE, ls='--', lw=1.2,
                       label=f'Class prevalence = {prevalence:.3f}')
            ax.set(xlabel='Recall', ylabel='Precision')
        ax.set(xlim=(0, 1), ylim=(0, 1))
        style_ax(ax, f'{prefix} — {label}: {curve_dropdown.value}')
        ax.legend(frameon=False, loc='lower left' if curve_dropdown.value == 'ROC' else 'upper right')
        plt.tight_layout()
        with chart:
            chart.clear_output(wait=True)
            display(fig)
        plt.close(fig)

    emotion_dropdown.observe(update, names='value')
    curve_dropdown.observe(update, names='value')
    display(HBox([emotion_dropdown, curve_dropdown]), chart)
    update()


val_scores = score_matrix(model_train, X_val, EMOTIONS)
validation_roc_pr = roc_pr_summary(y_val, val_scores, 'Validation')
explore_roc_pr(y_val, val_scores, 'Validation')

## 5. Model interpretation on validation
Permutation importance quantifies the decrease in validation Macro F1 when one feature is permuted; correlated features can share predictive signal and depress individual importance.

In [ ]:
# Held-out validation data only. Importance is model- and validation-dependent.
imp=permutation_importance(model_train,X_val,y_val,scoring='f1_macro',
                           n_repeats=8,random_state=SEED,n_jobs=-1)
importance_df=pd.DataFrame({'feature':FEATURE_COLS,'importance':imp.importances_mean,
                            'sd':imp.importances_std}).sort_values('importance',ascending=False)
importance_df.to_csv(RESULTS_DIR/'permutation_importance.csv',index=False)
show=importance_df.head(12).iloc[::-1]
fig,ax=plt.subplots(figsize=(9.8,5.2))
ax.barh(show.feature,show.importance,xerr=show.sd,color=PURPLE,
        error_kw={'ecolor':BLUE,'capsize':2},height=.68)
ax.axvline(0,color='#A0A0AA',lw=1)
ax.set(xlabel='Validation Macro F1 decrease after shuffling')
style_ax(ax,'Top acoustic features by permutation importance',grid='x')
plt.tight_layout();save_plot(fig,'permutation_importance.png')
display(importance_df.head(12).round(4))

## 6. Final frozen-model refit and test
The selected configuration is fitted to 18 actors before predicting actors 19–24. No further optimization is allowed after these results.

In [ ]:
# Single frozen-model refit on Train + Validation; test is used for evaluation only.
X_dev=pd.concat([X_train,X_val],ignore_index=True)
y_dev=pd.concat([y_train,y_val],ignore_index=True)
final_model=clone(model_train).fit(X_dev,y_dev)
X_test=df.loc[test_mask,FEATURE_COLS].reset_index(drop=True)
y_test=df.loc[test_mask,'emotion'].reset_index(drop=True)
test_pred=final_model.predict(X_test)
test_scores=score_matrix(final_model,X_test,EMOTIONS)
joblib.dump(final_model,RESULTS_DIR/'selected_pipeline_final.joblib')
test_metrics={'model':config['selected_model'],'test_accuracy':accuracy_score(y_test,test_pred),
    'test_balanced_accuracy':balanced_accuracy_score(y_test,test_pred),
    'test_macro_f1':f1_score(y_test,test_pred,labels=EMOTIONS,average='macro',zero_division=0),
    'test_weighted_f1':f1_score(y_test,test_pred,labels=EMOTIONS,average='weighted',zero_division=0)}
pd.DataFrame([test_metrics]).to_csv(RESULTS_DIR/'final_test_metrics.csv',index=False)
pred_cols=[c for c in ['relative_path','actor_id','emotion','intensity','statement','repetition'] if c in df.columns]
analysis=df.loc[test_mask,pred_cols].reset_index(drop=True).copy()
analysis['predicted_emotion']=test_pred
analysis['correct']=analysis.emotion.eq(analysis.predicted_emotion)
analysis.to_csv(RESULTS_DIR/'test_predictions.csv',index=False)
display(pd.DataFrame([test_metrics]).round(3))
print('Frozen pipeline evaluated without further tuning. Historical test inspection limits blind-test claims.')
# Descriptive threshold curves on the frozen test set; never used for tuning.
test_roc_pr = roc_pr_summary(y_test, test_scores, 'Test')
explore_roc_pr(y_test, test_scores, 'Test')


## 7. Test confusion patterns and class-wise results

In [ ]:
cm=confusion_matrix(y_test,test_pred,labels=EMOTIONS,normalize='true')
fig,ax=plt.subplots(figsize=(9,7))
sns.heatmap(cm,annot=True,fmt='.2f',cmap=SEQ,vmin=0,vmax=1,square=True,
            linewidths=.7,linecolor='white',xticklabels=EMOTIONS,yticklabels=EMOTIONS,
            cbar_kws={'label':'Recall by true emotion','shrink':.85},ax=ax)
ax.set(xlabel='Predicted emotion',ylabel='True emotion')
ax.tick_params(axis='both',length=0,labelrotation=0)
ax.set_title('Emotion confusions on unseen speakers',loc='left',fontweight='bold',pad=15)
plt.tight_layout();save_plot(fig,'test_confusion_matrix.png')
per_emotion=pd.DataFrame({'emotion':EMOTIONS,'n':[(y_test==e).sum() for e in EMOTIONS],
    'f1':[f1_score(y_test.eq(e),pd.Series(test_pred).eq(e),zero_division=0) for e in EMOTIONS]})
per_emotion.to_csv(RESULTS_DIR/'per_emotion_f1.csv',index=False)
fig,ax=plt.subplots(figsize=(9.3,4.7))
plot=per_emotion.sort_values('f1')
ax.barh(plot.emotion,plot.f1,color=PURPLE,height=.66)
ax.set(xlabel='One-vs-rest F1',xlim=(0,1))
style_ax(ax,'Which emotions are hardest to identify?',grid='x')
plt.tight_layout();save_plot(fig,'test_per_emotion_f1.png')
errors=analysis.loc[~analysis.correct].groupby(['emotion','predicted_emotion']).size().sort_values(ascending=False).rename('count').reset_index()
display(errors.head(8))
errors.to_csv(RESULTS_DIR/'error_pairs.csv',index=False)

## 8. Speaker, intensity and dataset voice groups
Intensity comparisons exclude neutral, which lacks strong-intensity recordings. Group comparisons are descriptive only.

In [ ]:
actor_table=pd.DataFrame([{'actor_id':a,'macro_f1':f1_score(sub.emotion,sub.predicted_emotion,labels=EMOTIONS,average='macro',zero_division=0),
                           'accuracy':sub.correct.mean()} for a,sub in analysis.groupby('actor_id')])
actor_table.to_csv(RESULTS_DIR/'actor_metrics.csv',index=False)
fig,ax=plt.subplots(figsize=(9.5,4.3))
ax.bar(actor_table.actor_id.map(lambda x:f'{x:02d}'),actor_table.macro_f1,width=.62,color=PURPLE)
ax.axhline(test_metrics['test_macro_f1'],color=BLUE,ls='--',lw=1.5,label='Overall Macro F1')
ax.set(xlabel='Held-out actor',ylabel='Macro F1',ylim=(0,1))
style_ax(ax,'Speaker variability on the held-out set')
ax.legend(frameon=False)
plt.tight_layout();save_plot(fig,'per_actor.png')
display(actor_table.round(3))
if 'intensity' in analysis.columns:
    intensity_map={'01':'Normal','02':'Strong','1':'Normal','2':'Strong','normal':'Normal','strong':'Strong'}
    intensity=analysis.copy()
    intensity['intensity_label']=intensity.intensity.astype(str).str.lower().map(intensity_map)
    matched=intensity.loc[intensity.emotion.ne('neutral')].copy()  # No strong neutral in RAVDESS.
    if matched.intensity_label.notna().all():
        tab=matched.groupby(['emotion','intensity_label']).correct.mean().unstack().reindex(
            index=[e for e in EMOTIONS if e!='neutral'],columns=['Normal','Strong'])
        fig,ax=plt.subplots(figsize=(10.3,4.3))
        pos=np.arange(len(tab))
        ax.plot(pos,tab.Normal,marker='o',color=BLUE,lw=2,label='Normal')
        ax.plot(pos,tab.Strong,marker='o',color=PURPLE,lw=2,label='Strong')
        ax.set(xticks=pos,xticklabels=tab.index,ylabel='Accuracy',ylim=(0,1))
        style_ax(ax,'Recognition by emotion intensity')
        ax.legend(frameon=False)
        plt.tight_layout();save_plot(fig,'intensity_accuracy.png')
        tab.to_csv(RESULTS_DIR/'intensity_accuracy.csv')
        print(f'Overall strong − normal accuracy (neutral excluded): {matched.loc[matched.intensity_label.eq("Strong"),"correct"].mean()-matched.loc[matched.intensity_label.eq("Normal"),"correct"].mean():+.3f}')
    else:print('Intensity labels unrecognized; intensity comparison skipped.')
analysis['voice_group']=np.where(analysis.actor_id.astype(int)%2==0,'Female','Male')
groups=analysis.groupby('voice_group').agg(n=('correct','size'),accuracy=('correct','mean'))
display(groups.round(3))
print('Voice-group differences are descriptive (three test actors per group), not population-level effects.')

## 9. Optional audio-noise robustness
Noise robustness requires reconstructing the exact pYIN-based feature extraction pipeline from notebook 03; it is not essential to the final supervised evaluation.

In [ ]:
RUN_NOISE_EXPERIMENT = False
NOISE_SNR_DB = 20  # Fixed in advance; not optimized using test outcomes.
NOISE_SAMPLES_PER_EMOTION = 4  # 32 recordings; four per emotion.
ZIP_PATH = PROJECT_DIR / 'data/Audio_Speech_Actors_01-24.zip'

if RUN_NOISE_EXPERIMENT:
    import io
    import zipfile
    import librosa
    from tqdm.auto import tqdm

    TARGET_SR, N_MFCC, N_FFT, HOP_LENGTH = 16000, 13, 1024, 256
    def features_from_signal(signal, sr=TARGET_SR):
        y, _ = librosa.effects.trim(signal, top_db=30)
        if len(y) == 0 or np.max(np.abs(y)) < 1e-6:
            raise ValueError('Empty or silent signal')
        features = {}
        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=N_MFCC,
                                    n_fft=N_FFT, hop_length=HOP_LENGTH)
        for i in range(N_MFCC):
            features[f'mfcc_{i+1:02d}_mean'] = float(mfcc[i].mean())
            features[f'mfcc_{i+1:02d}_std'] = float(mfcc[i].std())
        S = np.abs(librosa.stft(y, n_fft=N_FFT, hop_length=HOP_LENGTH))
        arrays = {
            'rms': librosa.feature.rms(y=y, frame_length=N_FFT, hop_length=HOP_LENGTH)[0],
            'zcr': librosa.feature.zero_crossing_rate(y, frame_length=N_FFT, hop_length=HOP_LENGTH)[0],
            'spectral_centroid': librosa.feature.spectral_centroid(S=S, sr=sr)[0],
            'spectral_bandwidth': librosa.feature.spectral_bandwidth(S=S, sr=sr)[0],
            'spectral_rolloff': librosa.feature.spectral_rolloff(S=S, sr=sr)[0],
        }
        for name, values in arrays.items():
            features[f'{name}_mean'] = float(values.mean())
            features[f'{name}_std'] = float(values.std())
        f0, voiced_flag, _ = librosa.pyin(
            y, sr=sr, fmin=65, fmax=500, frame_length=2048, hop_length=512
        )
        voiced = f0[np.isfinite(f0)]
        features['f0_hz_mean'] = float(voiced.mean()) if len(voiced) else 0.0
        features['f0_hz_std'] = float(voiced.std()) if len(voiced) else 0.0
        features['voiced_ratio'] = float(np.mean(voiced_flag))
        return features

    assert ZIP_PATH.is_file(), f'Original speech ZIP not found: {ZIP_PATH}'
    sample = (analysis.groupby('emotion', group_keys=False)
              .sample(n=NOISE_SAMPLES_PER_EMOTION, random_state=SEED)
              .sort_values('relative_path').reset_index(drop=True))
    rng = np.random.default_rng(SEED)
    rows = []
    with zipfile.ZipFile(ZIP_PATH) as archive:
        archive_names = archive.namelist()
        for record in tqdm(sample.itertuples(index=False), total=len(sample), desc='Noise robustness'):
            candidates = [name for name in archive_names if name.endswith(record.relative_path)]
            if len(candidates) != 1:
                raise FileNotFoundError(f'Expected 1 ZIP member for {record.relative_path}; got {len(candidates)}')
            with archive.open(candidates[0]) as f:
                audio, _ = librosa.load(io.BytesIO(f.read()), sr=TARGET_SR, mono=True)
            noise = rng.normal(size=len(audio))
            signal_rms = np.sqrt(np.mean(audio ** 2))
            noise_rms = np.sqrt(np.mean(noise ** 2))
            noisy = audio + noise * signal_rms / (noise_rms * 10 ** (NOISE_SNR_DB / 20))
            feat = features_from_signal(noisy)
            x_noise = pd.DataFrame([feat])[FEATURE_COLS]
            noisy_prediction = final_model.predict(x_noise)[0]
            rows.append({
                'relative_path': record.relative_path,
                'emotion': record.emotion,
                'clean_prediction': record.predicted_emotion,
                'noisy_prediction': noisy_prediction,
                'clean_correct': bool(record.correct),
                'noisy_correct': noisy_prediction == record.emotion,
            })

    noise_results = pd.DataFrame(rows)
    noise_results.to_csv(RESULTS_DIR / 'noise_paired_predictions.csv', index=False)
    robustness_summary = pd.DataFrame([
        {'Condition': 'Clean (matched subset)',
         'Accuracy': noise_results.clean_correct.mean(),
         'Macro F1': f1_score(noise_results.emotion, noise_results.clean_prediction,
                              labels=EMOTIONS, average='macro', zero_division=0)},
        {'Condition': f'White noise ({NOISE_SNR_DB} dB SNR)',
         'Accuracy': noise_results.noisy_correct.mean(),
         'Macro F1': f1_score(noise_results.emotion, noise_results.noisy_prediction,
                              labels=EMOTIONS, average='macro', zero_division=0)},
    ])
    robustness_summary.to_csv(RESULTS_DIR / 'noise_robustness_summary.csv', index=False)
    display(robustness_summary.round(3))
    print('Noisy − clean Macro F1:', round(robustness_summary.iloc[1]['Macro F1'] -
                                         robustness_summary.iloc[0]['Macro F1'], 3))
else:
    print('Optional noise robustness experiment not run.')

## 10. Conclusions and limitations

In [ ]:
print('FINAL RESEARCH FINDINGS')
print(f"Original validation Macro F1: {config['original_baseline_validation_f1']:.3f}; selected: {config['selected_validation_f1']:.3f}; delta {config['selected_validation_f1']-config['original_baseline_validation_f1']:+.3f}.")
print(f"Selected model: {config['selected_model']}; test Macro F1 {test_metrics['test_macro_f1']:.3f}; accuracy {test_metrics['test_accuracy']:.3f}.")
print(f'Validation macro ROC-AUC: {validation_roc_pr.ROC_AUC_OVR.mean():.3f}; average precision: {validation_roc_pr.average_precision.mean():.3f}.')
print('Lowest-F1 test emotions:', ', '.join(per_emotion.nsmallest(3,'f1').emotion))
if len(errors):
    bad=errors.iloc[0]
    print(f'Most common error: {bad.emotion} → {bad.predicted_emotion} ({bad["count"]} recordings).')
print(f'Per-actor Macro F1 range: {actor_table.macro_f1.min():.3f}–{actor_table.macro_f1.max():.3f}.')
print('Limitations: acted speech, six held-out speakers, historical test inspection, one dataset, no test-driven tuning.')

---
# Part II — V2: Temporal dynamics and trimming ablation

This section **preserves all original Notebook 05 analyses** while adding evaluation of the six feature tables from Notebook 03 V2 and 24 model/representation combinations from Notebook 04 V2.

**Methodological boundary:** The V2 exploratory candidate was chosen by speaker-grouped CV on actors 01–12. Actors 13–18 are a descriptive validation comparison; 19–24 have been inspected historically. Do **not** call a subsequent V2 test assessment a blind test, and do not tune on test outcomes.

**Fast execution:** In a fresh Colab session run original setup cells in §1 (imports/paths and style), then the following Part II cells. The slow original 05 analyses in §§2–10 can be retained as saved evidence without rerunning.

## 11. Verify V2 artifacts, schema, and selected experiment

Load exact Notebook 04 V2 CSV outputs and manifest, checking feature ordering, speaker split, recording alignment and recorded CV leader.

In [ ]:
# No audio extraction and no model fitting required to load V2 experiment results.
V2_ROOT = PROJECT_DIR / 'results/04_ml_v2'
V2_METRICS = V2_ROOT / 'metrics'
V2_SOURCE = PROJECT_DIR / 'data/processed/ravdess_v2_delta_rmsdiff_20261104'
V2_EVAL = PROJECT_DIR / 'results/05_evaluation_v2'
V2_FIGURES = V2_EVAL / 'figures'
V2_FIGURES.mkdir(parents=True, exist_ok=True)

required = [
    V2_METRICS / 'all_conditions.csv',
    V2_METRICS / 'grouped_cv_fold_scores.csv',
    V2_METRICS / 'exploratory_v2_manifest.json',
    V2_ROOT / 'exploratory_cv_candidate.joblib',
]
for path in required:
    if not path.is_file():
        raise FileNotFoundError(f'Missing Notebook 04 V2 artifact: {path}')

v2_summary = pd.read_csv(required[0])
v2_folds = pd.read_csv(required[1])
manifest = json.loads(required[2].read_text(encoding='utf-8'))
assert len(v2_summary) == 24 and len(v2_folds) == 72
assert set(v2_summary.representation) == {'static', 'enriched'}
assert set(v2_summary['mode']) == {'trim30', 'trim50', 'no_trim'}
assert set(manifest['train_actors']) == set(range(1,13))
assert set(manifest['validation_actors']) == set(range(13,19))
assert set(manifest['historically_inspected_test_actors']) == set(range(19,25))

chosen = manifest['chosen']
chosen_key = (chosen['mode'], chosen['representation'], chosen['model'])
chosen_row = v2_summary.loc[
    (v2_summary['mode'] == chosen_key[0]) &
    (v2_summary.representation == chosen_key[1]) &
    (v2_summary.model == chosen_key[2])
]
assert len(chosen_row) == 1
np.testing.assert_allclose(chosen_row.cv_macro_f1.to_numpy(), [chosen['cv_macro_f1']], rtol=0, atol=1e-12)
print('V2 24 conditions and 72 fold scores verified.')
print('CV-selected exploratory candidate:', chosen_key)
print('Grouped CV Macro F1:', round(chosen['cv_macro_f1'], 3))
print('Descriptive validation Macro F1:', round(chosen['validation_macro_f1_descriptive'],3))
display(v2_summary.sort_values('cv_macro_f1', ascending=False).head(10).round(3))

## 12. Compare the effects of temporal features, trimming, and generalization gaps

Show **both CV and validation** rather than cherry-picking one; keep the earlier validation-selected baseline alongside the new exploratory results.

In [ ]:
# Paired CV differences are grouped by identical actor folds.
paired = v2_folds.pivot(index=['mode','model','fold'], columns='representation',values='cv_f1')
assert paired.shape == (36,2)
paired['delta'] = paired['enriched'] - paired['static']
effects = paired.groupby('mode')['delta'].agg(['mean','std','min','max']).reset_index()
display(effects.round(3))

fig, axes = plt.subplots(1,3, figsize=(17,4.6))
for ax,metric,title in zip(axes[:2],['cv_macro_f1','validation_macro_f1'],
                            ['Speaker-grouped CV Macro F1','Validation Macro F1 (descriptive)']):
    view=v2_summary.copy()
    view['condition']=view['mode']+' / '+view['representation']
    order=[m+' / '+r for m in ['trim30','trim50','no_trim'] for r in ['static','enriched']]
    pv=view.pivot(index='model',columns='condition',values=metric).reindex(columns=order)
    sns.heatmap(pv, annot=True, fmt='.3f', cmap=SEQ, vmin=0.25,vmax=0.50,
                linewidths=.5,linecolor='white',ax=ax,cbar=False)
    ax.set_title(title,loc='left',fontweight='bold',pad=12)
    ax.set(xlabel='',ylabel='')
    ax.tick_params(axis='x',rotation=40)

sns.boxplot(data=paired.reset_index(),x='mode',y='delta',color='#E4CBF2',ax=axes[2])
sns.stripplot(data=paired.reset_index(),x='mode',y='delta',hue='model',
              dodge=False, size=4,ax=axes[2])
axes[2].axhline(0,ls='--',lw=1,color=BLUE)
axes[2].set(title='Enriched − static CV F1 (paired folds)',xlabel='',ylabel='Difference')
axes[2].legend(title='Model',fontsize=7)
plt.tight_layout()
fig.savefig(V2_FIGURES/'v2_research_synthesis.png',dpi=170,bbox_inches='tight')
plt.show()

original_validation_f1 = float(config['selected_validation_f1'])
cv_leader_validation_f1 = float(chosen_row.validation_macro_f1.iloc[0])
validation_leader=v2_summary.loc[v2_summary.validation_macro_f1.idxmax()]
print(f'Original frozen validation F1: {original_validation_f1:.3f}')
print(f'V2 grouped-CV leader validation F1: {cv_leader_validation_f1:.3f}')
print(f'V2 validation leader (DESCRIPTIVE, not CV-selected): {validation_leader.model}, '
      f'{validation_leader["mode"]}/{validation_leader.representation}: '
      f'{validation_leader.validation_macro_f1:.3f} (CV {validation_leader.cv_macro_f1:.3f})')


## 13. Validate the saved CV-selected candidate on the held-out validation actors

Re-load its exact feature table and saved model to reproduce the recorded validation score. Plot per-class F1 and confusion matrix. No model selection is made here.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
candidate = joblib.load(V2_ROOT/'exploratory_cv_candidate.joblib')  # Load only your own model.
feature_path = V2_SOURCE / f'features_{chosen["mode"]}_{chosen["representation"]}.csv'
if not feature_path.is_file():
    raise FileNotFoundError(feature_path)
feature_table=pd.read_csv(feature_path).sort_values('relative_path').reset_index(drop=True)
assert feature_table.relative_path.equals(df.relative_path)
assert feature_table.emotion.astype(str).equals(df.emotion.astype(str))
assert feature_table.actor_id.astype(int).equals(df.actor_id.astype(int))
columns=manifest['feature_names']
assert len(columns)==len(set(columns)) and set(columns).issubset(feature_table.columns)
assert len(columns)==int(chosen_row.n_features.iloc[0])
assert np.isfinite(feature_table[columns].to_numpy(dtype=float)).all()
val_features=feature_table.loc[val_mask,columns].reset_index(drop=True)
val_truth=feature_table.loc[val_mask,'emotion'].reset_index(drop=True)
val_prediction=candidate.predict(val_features)
val_score=f1_score(val_truth,val_prediction,labels=EMOTIONS,average='macro',zero_division=0)
np.testing.assert_allclose(val_score,float(chosen_row.validation_macro_f1.iloc[0]),atol=1e-12,rtol=0)

report=classification_report(val_truth,val_prediction,labels=EMOTIONS,output_dict=True,zero_division=0)
per_emotion_v2=pd.DataFrame([{'emotion':e,'precision':report[e]['precision'],
                             'recall':report[e]['recall'],'f1':report[e]['f1-score'],
                             'support':report[e]['support']} for e in EMOTIONS])
per_emotion_v2.to_csv(V2_EVAL/'cv_selected_validation_per_emotion.csv',index=False)
fig,axes=plt.subplots(1,2,figsize=(14,5.3))
cm=confusion_matrix(val_truth,val_prediction,labels=EMOTIONS,normalize='true')
sns.heatmap(cm,cmap=SEQ,vmin=0,vmax=1,annot=True,fmt='.2f',
            xticklabels=EMOTIONS,yticklabels=EMOTIONS,ax=axes[0])
axes[0].set(xlabel='Predicted emotion',ylabel='True emotion',title='CV-selected V2: validation confusion')
axes[0].tick_params(axis='x',rotation=45)
sns.barplot(data=per_emotion_v2,y='emotion',x='f1',color=PURPLE,ax=axes[1])
axes[1].set(xlim=(0,1),xlabel='F1',ylabel='',title='Validation F1 by emotion')
plt.tight_layout(); fig.savefig(V2_FIGURES/'cv_selected_validation_diagnostics.png',dpi=170,bbox_inches='tight');plt.show()
print(f'Reproduced CV-selected V2 validation Macro F1: {val_score:.3f}')
display(per_emotion_v2.round(3))

## 14. Optional, explicitly labeled **historical-test** descriptive comparison

**Default: OFF.** Actors 19–24 were already inspected in previous versions, so these results cannot be characterized as an independent blind evaluation. Enable only if you have decided to report an exploratory comparison without further model selection or parameter adjustment.

If enabled, refit the **preselected CV winner** on actors 01–18 and evaluate actors 19–24 once for descriptive comparison with the prior frozen model. Do not use the outcome to choose a different model.

In [ ]:
RUN_V2_HISTORICAL_TEST = False  # Opt in only for descriptive reporting, never for tuning.
if RUN_V2_HISTORICAL_TEST:
    development_mask = train_mask | val_mask
    historical_test_mask = test_mask
    Xt_dev = feature_table.loc[development_mask,columns].reset_index(drop=True)
    yt_dev = feature_table.loc[development_mask,'emotion'].reset_index(drop=True)
    Xt_test = feature_table.loc[historical_test_mask,columns].reset_index(drop=True)
    yt_test = feature_table.loc[historical_test_mask,'emotion'].reset_index(drop=True)
    final_v2 = clone(candidate).fit(Xt_dev,yt_dev)
    historical_pred = final_v2.predict(Xt_test)
    historical_metrics = {
        'model': chosen['model'], 'mode':chosen['mode'],
        'representation':chosen['representation'],
        'test_macro_f1_descriptive':f1_score(yt_test,historical_pred,labels=EMOTIONS,average='macro',zero_division=0),
        'test_accuracy_descriptive':accuracy_score(yt_test,historical_pred),
        'test_balanced_accuracy_descriptive':balanced_accuracy_score(yt_test,historical_pred),
        'status':'historical test reused; NOT blind or tuning evidence'
    }
    pd.DataFrame([historical_metrics]).to_csv(V2_EVAL/'historical_test_descriptive.csv',index=False)
    joblib.dump(final_v2,V2_EVAL/'cv_selected_refit_dev_model.joblib')
    display(pd.DataFrame([historical_metrics]).round(3))
    print('Do not tune or reselect after seeing these historical test results.')
else:
    print('Historical test intentionally not run. Validation and CV analysis are ready.')

## 15. Interpretation for the report

- **H1 — Dynamics:** For fixed model and fold, compare `enriched − static`; effects may vary by trimming and speaker group. This is an exploratory empirical finding, not causality.
- **H2 — Trimming:** Compare 30 dB vs 50 dB vs no trimming under the same acoustic representation and model; more retained audio is **not** automatically better.
- **H3 — Generalization:** Interpret the gap between training, grouped CV and validation; strong differences across disjoint actor sets mean the estimate is unstable.
- **Preserved prior work:** earlier PCA, clustering, ablations, regularization, learning curves, ROC/PR, permutation importance, per-speaker and class-wise analyses remain above as the complete methodological progression.
- **Limitations:** RAVDESS is acted speech; actor group splits are small; V2 was chosen from multiple exploratory comparisons; previous inspection means actors 19–24 are not truly blind. Never describe the best validation condition as the CV-selected condition.

**Next:** Create Notebook 06 for cross-experiment conclusions and transparent research write-up. No new audio extraction is necessary.